# Travel Style Classifier — ML Pipeline

**Purpose:** EDA → feature engineering → 3-model CV comparison → LightGBM tuning → save best pipeline.

**Labels:** Adventure | Relaxation | Culture | Budget | Luxury | Family

**No-leakage rule:** `destination` and `country` columns are dropped before any fitting.
All preprocessing lives inside the `sklearn.Pipeline` so it is fit only on training folds.

**Reproducibility:** seeds fixed at 42 throughout.

In [ ]:
import importlib.metadata, platform, sys
for pkg in ["scikit-learn", "lightgbm", "pandas", "numpy", "matplotlib", "seaborn", "joblib"]:
    try:
        print(f"{pkg}=={importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{pkg}=NOT INSTALLED")
print(f"python=={sys.version}")
print(f"platform=={platform.platform()}")

In [ ]:
import random
import json
import csv
from pathlib import Path
from datetime import UTC, datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

ML_DIR = Path(".")
DATA_PATH = ML_DIR / "destinations.csv"
MODEL_PATH = ML_DIR / "model.joblib"
META_PATH  = ML_DIR / "model_meta.json"
RESULTS_PATH = ML_DIR / "results.csv"
CONFIDENCE_THRESHOLD = 0.60

TARGET_COL = "travel_style"
DROP_COLS  = ["destination", "country"]

NUMERIC_COLS = [
    "avg_temp_peak_season_c", "peak_season_length_months", "unesco_sites_count",
    "outdoor_activity_score", "daily_cost_bucket", "coastal_access",
    "visa_difficulty", "english_prevalence",
]
CATEGORICAL_COLS = ["climate_zone", "terrain_primary", "accommodation_range", "tourism_maturity"]

print("Setup complete.")

## 1. Load & Explore Data

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head()

In [ ]:
print("Missing values per column:")
print(df_raw.isnull().sum())
print("\nData types:")
print(df_raw.dtypes)

In [ ]:
# Class distribution — key for understanding imbalance
fig, ax = plt.subplots(figsize=(8, 4))
counts = df_raw[TARGET_COL].value_counts()
ax.bar(counts.index, counts.values, color=sns.color_palette("Set2", len(counts)))
ax.set_title("Class Distribution (travel_style)")
ax.set_xlabel("Travel Style")
ax.set_ylabel("Count")
for i, (label, cnt) in enumerate(counts.items()):
    ax.text(i, cnt + 0.5, str(cnt), ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()
print(counts.to_string())

In [ ]:
# Numeric feature distributions by travel style
drop = [c for c in DROP_COLS if c in df_raw.columns]
df = df_raw.drop(columns=drop)
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, NUMERIC_COLS):
    for style in sorted(y.unique()):
        ax.hist(X.loc[y == style, col], alpha=0.5, label=style, bins=12)
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.suptitle("Numeric Feature Distributions by Travel Style", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Categorical feature counts by travel style
fig, axes = plt.subplots(1, len(CATEGORICAL_COLS), figsize=(20, 5))
for ax, col in zip(axes, CATEGORICAL_COLS):
    ct = pd.crosstab(df[col], df[TARGET_COL])
    ct.plot(kind="bar", ax=ax, legend=(col == CATEGORICAL_COLS[0]))
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("Categorical Feature Counts by Travel Style", y=1.02)
plt.tight_layout()
plt.show()

## 2. Pipeline Construction

All preprocessing is encapsulated inside a single `sklearn.Pipeline`.
`StandardScaler` for numerics, `OneHotEncoder(handle_unknown="ignore")` for categoricals.
The `ColumnTransformer` drops any column not listed in `NUMERIC_COLS` or `CATEGORICAL_COLS`,
so destination name / country cannot leak through accidental inclusion.

In [ ]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), NUMERIC_COLS),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
        ],
        remainder="drop",
    )

def make_pipeline(clf):
    return Pipeline([("preprocess", make_preprocessor()), ("clf", clf)])

print("Pipeline factory defined.")

## 3. Three-Model Cross-Validation Comparison

5-fold `StratifiedKFold` — stratification preserves per-class proportions across folds,
which matters when some styles are rare. All models use `class_weight="balanced"` to
counter class imbalance without discarding data.

In [ ]:
CV_FOLDS = 5
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

models = {
    "LogisticRegression": LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
    "RandomForest":       RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "LightGBM":           LGBMClassifier(class_weight="balanced", random_state=RANDOM_STATE, verbose=-1),
}

cv_results = {}
for name, clf in models.items():
    pipe = make_pipeline(clf)
    scores = cross_validate(pipe, X, y, cv=cv, scoring=["accuracy", "f1_macro"], return_train_score=False)
    acc_mean, acc_std = np.mean(scores["test_accuracy"]), np.std(scores["test_accuracy"])
    f1_mean, f1_std   = np.mean(scores["test_f1_macro"]),  np.std(scores["test_f1_macro"])
    cv_results[name] = {"acc_mean": acc_mean, "acc_std": acc_std, "f1_mean": f1_mean, "f1_std": f1_std}
    print(f"{name:25s}  acc={acc_mean:.4f}±{acc_std:.4f}  macro-F1={f1_mean:.4f}±{f1_std:.4f}")

In [ ]:
# Visualise CV comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
names = list(cv_results.keys())
for ax, metric, label in [
    (axes[0], "acc_mean",  "CV Accuracy (mean ± std)"),
    (axes[1], "f1_mean",   "CV Macro-F1 (mean ± std)"),
]:
    means = [cv_results[n][metric] for n in names]
    stds  = [cv_results[n][metric.replace("mean", "std")] for n in names]
    ax.bar(names, means, yerr=stds, capsize=5, color=sns.color_palette("Set2", len(names)))
    ax.set_title(label)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class precision / recall / F1 on a held-out split
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print("Per-class classification reports (held-out 20%):\n")
for name, clf in models.items():
    pipe = make_pipeline(clf)
    pipe.fit(X_tr, y_tr)
    print(f"── {name} ──")
    print(classification_report(y_val, pipe.predict(X_val), zero_division=0))
    print()

## 4. LightGBM Hyperparameter Tuning

`RandomizedSearchCV(n_iter=30, scoring="f1_macro")` — macro-F1 as the objective
because accuracy would reward predicting the majority class.

**Why these hyperparameters:**
- `num_leaves` / `max_depth` — control model complexity; the main lever against overfitting on 205 rows
- `learning_rate` × `n_estimators` — trade off bias and variance; slower rate needs more trees
- `min_child_samples` — minimum samples per leaf; directly mitigates overfitting on small classes
- `reg_alpha` — L1 regularisation; additional protection for small datasets

In [ ]:
param_distributions = {
    "clf__num_leaves":        [15, 31, 63],
    "clf__max_depth":         [-1, 4, 6, 8],
    "clf__learning_rate":     [0.01, 0.05, 0.1],
    "clf__n_estimators":      [100, 200, 400],
    "clf__min_child_samples": [5, 10, 20],
    "clf__reg_alpha":         [0.0, 0.1, 1.0],
}

lgbm_pipe = make_pipeline(LGBMClassifier(class_weight="balanced", random_state=RANDOM_STATE, verbose=-1))
search = RandomizedSearchCV(
    lgbm_pipe, param_distributions,
    n_iter=30, cv=cv, scoring="f1_macro",
    random_state=RANDOM_STATE, n_jobs=-1, refit=True,
)
search.fit(X, y)

print(f"Best macro-F1 : {search.best_score_:.4f}")
print(f"Best params   : {search.best_params_}")

## 5. Select Winner & Confusion Matrix

In [ ]:
all_candidates = {**{n: cv_results[n]["f1_mean"] for n in cv_results}, "LightGBM_tuned": search.best_score_}
winner_name = max(all_candidates, key=all_candidates.__getitem__)
print(f"Winner: {winner_name}  (macro-F1={all_candidates[winner_name]:.4f})")
print("\nAll candidates:")
for n, f1 in sorted(all_candidates.items(), key=lambda x: -x[1]):
    print(f"  {n:25s} {f1:.4f}")

In [ ]:
# Confusion matrix on held-out 20%
winner_pipe = search.best_estimator_ if winner_name == "LightGBM_tuned" else make_pipeline(models[winner_name])
winner_pipe.fit(X_tr, y_tr)
y_pred_val = winner_pipe.predict(X_val)
classes = sorted(y.unique())

cm = confusion_matrix(y_val, y_pred_val, labels=classes)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes, ax=ax, cmap="Blues")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — {winner_name}")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print("Saved confusion_matrix.png")

## 6. Refit on Full Dataset & Save

CV is complete — we now refit the winner on **all 205 rows** to maximise the signal
going into the production model. The pipeline is serialised with `joblib` alongside
a `model_meta.json` sidecar that stores the confidence threshold, class list,
feature columns, and timestamp so the serving code never needs to re-derive them.

In [ ]:
# Refit on full dataset
winner_pipe_full = search.best_estimator_ if winner_name == "LightGBM_tuned" else make_pipeline(models[winner_name])
winner_pipe_full.fit(X, y)

joblib.dump(winner_pipe_full, MODEL_PATH)
print(f"Model saved → {MODEL_PATH}")

meta = {
    "model": winner_name,
    "classes": classes,
    "feature_columns": NUMERIC_COLS + CATEGORICAL_COLS,
    "threshold": CONFIDENCE_THRESHOLD,
    "timestamp": datetime.now(tz=UTC).isoformat(),
}
META_PATH.write_text(json.dumps(meta, indent=2))
print(f"Meta  saved → {META_PATH}")
print(json.dumps(meta, indent=2))

In [ ]:
# Smoke test: one sample prediction
sample = X.iloc[[0]]
proba = winner_pipe_full.predict_proba(sample)[0]
pred_class = winner_pipe_full.classes_[np.argmax(proba)]
confidence = float(np.max(proba))
print(f"Sample: {df_raw.iloc[0]['destination'] if 'destination' in df_raw.columns else 'row 0'}")
print(f"True label : {y.iloc[0]}")
print(f"Predicted  : {pred_class}  (confidence={confidence:.4f})")
print(f"All probs  : {dict(zip(winner_pipe_full.classes_, proba.round(4)))}")
assert abs(sum(proba) - 1.0) < 1e-6, "probabilities must sum to 1"
print("Smoke test passed ✓")